<a href="https://colab.research.google.com/github/legna7816/ml-projects/blob/main/rag/rag_step02_Generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# !pip install transformers torch accelerate sentence-transformers # 최초 1회 설치

import torch
import numpy as np
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('사용 디바이스:', device)

사용 디바이스: cuda


In [2]:
# 1. 검색 (Retrieval) 파트
embed_model = SentenceTransformer('jhgan/ko-sroberta-multitask')

documents = [
    "타이타닉은 1912년 4월 15일 빙산과 충돌해 침몰한 영국의 여객선이다.",
    "파이썬은 1991년 귀도 반 로섬이 개발한 프로그래밍 언어이다.",
    "BERT는 구글이 2018년에 발표한 자연어처리 모델이다.",
    "김치는 발효 채소를 이용한 한국의 전통 음식이다.",
    "RAG는 검색과 생성을 결합한 자연어처리 기법이다.",
    "딥러닝은 인공신경망을 여러 층으로 쌓아 학습하는 머신러닝의 한 분야이다.",
    "된장은 콩으로 만든 메주를 소금물에 발효시켜 만든다.",
    "컴퓨터는 사람이 명령을 내리면 인간보다 훨씬 빠르고 정확하게 계산을 대신해 주는 기계이다.",
    "핸드폰은 컴퓨터를 손바닥만 한 크기로 줄여서 들고 다닐 수 있게 만든 무선 통신 기기이다.",
]
doc_embeddings = embed_model.encode(documents)

def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def search(query, top_k=2):
    query_vec = embed_model.encode(query)
    scores = [cosine_sim(query_vec, doc_vec) for doc_vec in doc_embeddings]
    top_indices = np.argsort(scores)[::-1][:top_k]
    return [documents[i] for i in top_indices]   # 이제 문서 텍스트만 반환


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/4.86k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/744 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/585 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/248k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/495k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [5]:
# 2. 생성 (Generation) 모델 불러오기 (가볍고 한국어 되는 모델)
# Qwen2.5-1.5B-Insruct: 알리바바가 공개한 무료 오픈소스 모델 (한국어도 지원)
# "Instruct"가 붙은 건 "질문에 답하도록" 추가 학습된 버전 (일반 텍스트 생성용보다 대화에 적합)
gen_model_name = "Qwen/Qwen2.5-1.5B-Instruct"
gen_tokenizer = AutoTokenizer.from_pretrained(gen_model_name)
gen_model = AutoModelForCausalLM.from_pretrained(
    gen_model_name,
    torch_dtype=torch.float16,  # 메모리 절약을 위해 16비트로 불러오기
    device_map='auto'
)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [8]:
import torch
import numpy as np
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM

# 3, RAG 파이프라인 (검색 + 생성을 하나로 연결)
def rag_answer(query):
  # (1) 검색: 질문과 관련된 문서 찾기
  retrieved_docs = search(query, top_k=2)
  context = "\n".join(retrieved_docs)

  # (2) 프롬프트 구성: 검색된 문서를 "참고 자료"로 LLM에 제공
  prompt = f"""다음 참고 자료를 바탕으로 질문에 답하세요. 참고 자료에 없는 내용은 답하지 마세요,

참고 자료:
{context}

질문: {query}
답변:"""

  # (3) 생성: LLM에 참고 자료를 보고 답변 생성
  messages = [{"role": "user", "content": prompt}]
  text = gen_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
  inputs = gen_tokenizer(text, return_tensors="pt").to(device)

  with torch.no_grad():
    outputs = gen_model.generate(
        **inputs,
        max_new_tokens=150,  # 최대 답변 길이
        temperature=0.7,     # 낮을수록 일관된 답변, 높을수록 다양한 답변
        do_sample=True
    )
  response = gen_tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

  return retrieved_docs, response

In [9]:
# 4. 실제 RAG 테스트
query = "타이타닉은 언제 침몰했어?"
docs, answer = rag_answer(query)

print("질문:", query)
print("\n[검색된 참고 자료]")
for d in docs:
  print(" -", d)
print("\n[생성된 답변]")
print(answer)

질문: 타이타닉은 언제 침몰했어?

[검색된 참고 자료]
 - 타이타닉은 1912년 4월 15일 빙산과 충돌해 침몰한 영국의 여객선이다.
 - 김치는 발효 채소를 이용한 한국의 전통 음식이다.

[생성된 답변]
타이타닉은 1912년 4월 15일 침몰했습니다.


In [13]:
# 5. TODO
# 5-1. "RAG가 뭐야?"라는 질문으로 rag_answer() 실행 후 결과 확인
query1 = "RAG가 뭐야?"
docs1, answer1 = rag_answer(query1)

print("질문:", query1)
print("\n[검색된 참고 자료]")
for d in docs1:
  print(" -", d)
print("\n[생성된 답변]")
print(answer1)

질문: RAG가 뭐야?

[검색된 참고 자료]
 - RAG는 검색과 생성을 결합한 자연어처리 기법이다.
 - BERT는 구글이 2018년에 발표한 자연어처리 모델이다.

[생성된 답변]
RAG는 검색과 생성을 결합한 자연어 처리 기법입니다.


In [14]:
# 5-2. documents에 없는 내용 질문 (예: "아이폰은 언제 나왔어?") 던져보고
# "참고 자료에 없는 내용은 답하지 마세요" 지시가 실제로 지켜지는지 확인
query2 = "아이폰은 언제 나왔어?"
docs2, answer2 = rag_answer(query2)

print("질문:", query2)
print("\n[검색된 참고 자료]")
for d in docs2:
  print(" -", d)
print("\n[생성된 답변]")
print(answer2)

질문: 아이폰은 언제 나왔어?

[검색된 참고 자료]
 - 핸드폰은 컴퓨터를 손바닥만 한 크기로 줄여서 들고 다닐 수 있게 만든 무선 통신 기기이다.
 - BERT는 구글이 2018년에 발표한 자연어처리 모델이다.

[생성된 답변]
아이폰은 2007년에 출시되었습니다.


In [16]:
# 5-3. rag_answer() 없이, context 없이 LLM에 그냥 질문만 던졌을 때랑 결과 비교
def plain_answer(query):
  messages = [{"role": "user", "content": query}]
  text = gen_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
  inputs = gen_tokenizer(text, return_tensors="pt").to(device)

  with torch.no_grad():
    outputs = gen_model.generate(
        **inputs,
        max_new_tokens=150,  # 최대 답변 길이
        temperature=0.7,     # 낮을수록 일관된 답변, 높을수록 다양한 답변
        do_sample=True
    )
  response = gen_tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

  return response

query = "RAG는 뭐야?"
# RAG 있을 때
docs, rag_result = rag_answer(query)
print("=== RAG 사용 ===")
print("참고 자료:", docs)
print("답변:", rag_result)

print()

# RAG 없을 때
plain_result = plain_answer(query)
print("=== RAG 미사용 ===")
print("답변:", plain_result)

=== RAG 사용 ===
참고 자료: ['RAG는 검색과 생성을 결합한 자연어처리 기법이다.', 'BERT는 구글이 2018년에 발표한 자연어처리 모델이다.']
답변: RAG는 검색과 생성을 결합한 자연어 처리 기법입니다.

=== RAG 미사용 ===
답변: "RAG"은 "Relevance-Aware Gating"의 약자로, 자연어 처리(NLP) 분야에서 주요한 기술입니다.

RAG는 다음과 같은 중요한 특징을 가집니다:

1. 시각적 정보를 포함하는 대화를 이해하기 위한 방법: RAG는 대화 상대방이 어떤 내용을 제안하거나 질문을 하는지에 대한 정보를 포함하는 대화를 이해하는데 도움을 줍니다.

2. 문맥과 관련성: RAG는 문맥에 따라 최선의 답변을 제공하도록 설계되어 있습니다. 이는 일반적으로 문서 검색이나 인공 지능(AI) 모델에서 사용되는 기


In [18]:
documents_fake = [
    "1+1은 3이다.",
    "세종대왕은 조선의 왕이 아니라 고려의 왕이었다.",
    "김치는 한국이 아니라 중국에서 기원한 음식이다.",
]
doc_embeddings_fake = embed_model.encode(documents_fake)

def search_fake(query, top_k=1):
    query_vec = embed_model.encode(query)
    scores = [cosine_sim(query_vec, doc_vec) for doc_vec in doc_embeddings_fake]
    top_indices = np.argsort(scores)[::-1][:top_k]
    return [documents_fake[i] for i in top_indices]

def rag_answer_fake(query):
    retrieved_docs = search_fake(query, top_k=1)
    context = "\n".join(retrieved_docs)
    prompt = f"""다음 참고 자료를 바탕으로 질문에 답하세요. 참고 자료에 없는 내용은 답하지 마세요.

참고 자료:
{context}

질문: {query}
답변:"""
    messages = [{"role": "user", "content": prompt}]
    text = gen_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = gen_tokenizer(text, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = gen_model.generate(**inputs, max_new_tokens=100, temperature=0.7, do_sample=True)
    response = gen_tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return retrieved_docs, response


# 질문 3개로 각각 테스트
docs1, ans1 = rag_answer_fake("1+1은 뭐야?")
print("참고 자료:", docs1)
print("답변:", ans1)
print()

docs2, ans2 = rag_answer_fake("세종대왕은 어느 나라의 왕이었어?")
print("참고 자료:", docs2)
print("답변:", ans2)
print()

docs3, ans3 = rag_answer_fake("김치는 어느 나라 음식이야?")
print("참고 자료:", docs3)
print("답변:", ans3)

참고 자료: ['1+1은 3이다.']
답변: 1+1은 2입니다.

참고 자료: ['세종대왕은 조선의 왕이 아니라 고려의 왕이었다.']
답변: 세종대왕은 고려의 왕이었습니다.

참고 자료: ['김치는 한국이 아니라 중국에서 기원한 음식이다.']
답변: 김치는 중국에서 기원한 음식입니다.
